necessary libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

loading the interpolated data

In [2]:
DATASETS = Path("/Users/muskaanchugh/Desktop/art_synthetic_data_gen/datasets")
DATASETS.mkdir(exist_ok=True)

INTERPOLATED = Path("/Users/muskaanchugh/Desktop/art_synthetic_data_gen/interpolated_data")
interpolated_dfs = {f.stem.replace("_synthetic_interpolated", ""): pd.read_csv(f) for f in INTERPOLATED.glob("*.csv")}

executor function for introduction of noise and spikes for a real world simulation

In [3]:
def apply_noise_and_spikes(df, config):
    np.random.seed(config.get("seed", 42))
    patients = df["patients"].values.astype(float)
    n = len(patients)

    # generating noise
    noise = np.zeros(n)
    for phase in config["noise_phases"]:
        start = phase["start"]
        end = phase["end"] if phase["end"] is not None else n
        noise[start:end] = np.random.normal(0, patients[start:end] * phase["fraction"])

    # applying noise first if configured (lpv_pediatric)
    if config.get("noise_first", False):
        patients = np.clip(patients + noise, 0, None)
        noise = np.zeros(n)

    # generating spikes
    spikes = np.zeros(n)
    for spike in config.get("spikes", []):
        indices = spike["indices"]
        if indices == "random":
            indices = np.random.choice(n, size=spike["count"], replace=False)
        for idx in indices:
            if idx >= n:
                continue
            if spike["direction"] == "random":
                d = np.random.choice([-1, 1])
            elif spike["direction"] == "positive":
                d = 1
            else:
                d = -1
            spikes[idx] += d * np.random.randint(spike["min"], spike["max"])

    result = df.copy()
    result["patients"] = np.clip(np.round(patients + noise + spikes), 0, None).astype(int)
    return result

configuration for noise and spikes for each drug

In [4]:
configs = {
    "abc_3tc_adult": {
        "seed": 42,
        "noise_phases": [
            {"start": 0,  "end": 41,   "fraction": 0.025},
            {"start": 41, "end": 77,   "fraction": 0.02},
            {"start": 77, "end": None, "fraction": 0.015}
        ],
        "spikes": [
            {"indices": [11, 48, 77], "direction": "random", "min": 1500, "max": 2200}
        ]
    },
    "atv_adult": {
        "seed": 42,
        "noise_phases": [
            {"start": 0,  "end": 53,   "fraction": 0.03},
            {"start": 53, "end": None, "fraction": 0.06}
        ],
        "spikes": [
            {"indices": [94], "direction": "negative", "min": 2000, "max": 4000}
        ]
    },
    "azt_3tc_adult": {
        "seed": 42,
        "noise_phases": [
            {"start": 0,  "end": 41,   "fraction": 0.04},
            {"start": 41, "end": 77,   "fraction": 0.025},
            {"start": 77, "end": None, "fraction": 0.05}
        ],
        "spikes": [
            {"indices": "random", "count": 5, "direction": "positive", "min": 1500, "max": 3000}
        ]
    },
    "drv_adult": {
        "seed": 42,
        "noise_phases": [
            {"start": 0, "end": None, "fraction": 0.05}
        ],
        "spikes": [
            {"indices": [13, 58, 83], "direction": "random", "min": 200, "max": 800}
        ]
    },
    "abc_3tc_pediatric": {
        "seed": 42,
        "noise_phases": [
            {"start": 0,  "end": 41,   "fraction": 0.025},
            {"start": 41, "end": 77,   "fraction": 0.03},
            {"start": 77, "end": None, "fraction": 0.06}
        ],
        "spikes": [
            {"indices": [56, 82],             "direction": "random", "min": 3000, "max": 8000},
            {"indices": [77, 78, 80, 81, 84], "direction": "random", "min": 1000, "max": 2000}
        ]
    },
    "azt_3tc_pediatric": {
        "seed": 42,
        "noise_phases": [
            {"start": 0,  "end": 41,   "fraction": 0.03},
            {"start": 41, "end": 89,   "fraction": 0.05},
            {"start": 89, "end": None, "fraction": 0.08}
        ],
        "spikes": [
            {"indices": [8, 56, 82],    "direction": "random", "min": 2000, "max": 5000},
            {"indices": [80, 81, 82, 84], "direction": "random", "min": 1000, "max": 2000}
        ]
    },
    "lpv_pediatric": {
        "seed": 42,
        "noise_first": True,
        "noise_phases": [
            {"start": 0, "end": None, "fraction": 0.05}
        ],
        "spikes": [
            {"indices": [8, 55],          "direction": "random", "min": 3000, "max": 6000},
            {"indices": [77, 78, 81, 83], "direction": "random", "min": 1000, "max": 2000}
        ]
    }
}

running executor function with given configs for each drug

In [5]:
noisy_dfs = {}

for name, config in configs.items():
    df = interpolated_dfs[name]
    noisy_dfs[name] = apply_noise_and_spikes(df, config)

In [6]:
for name, df in noisy_dfs.items():
    df.to_csv(DATASETS / f"{name}_pre_calibration.csv", index=False)